### Overview

Získané výsledky ukazujú, že v prípade KNN výber hyperparametrov nie vždy vedie k zlepšeniu kvality modelu, zatiaľ čo zložitejšie prístupy, ako je DTW, môžu byť pre daný typ údajov menej efektívne.
Zo všetkých variantov KNN sa ako najlepší ukázal základný model, ktorý dosiahol najvyššiu hodnotu F1-miery a zabezpečil stabilnú rovnováhu medzi presnosťou a spätnou väzbou. Najhoršie výsledky dosiahol model využívajúci metriku DTW, ktorý mal výrazne nižšiu hodnotu F1-miery, čo svedčí o jeho neefektívnosti pre danú úlohu.

In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from tslearn.neighbors import KNeighborsTimeSeriesClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix)

!pip install tslearn

In [4]:
TW_500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=500/Twitter-Relative-Sigma-500.data",
    sep=",",
    header=None
)

TW_1000= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1000/Twitter-Relative-Sigma-1000.data",
    sep=",",
    header=None
)
TW_1500= pd.read_csv(
    "../classification/Twitter/Relative_labeling/sigma=1500/Twitter-Relative-Sigma-1500.data",
    sep=",",
    header=None
)
groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label") 

TW_500.columns = columns
TW_1000.columns = columns
TW_1500.columns = columns


### 500

### Baseline KNN

In [3]:
X = TW_500.drop(columns=['label'])
y = TW_500['label']

X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,stratify=y,random_state=42)

pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

pipe_knn.fit(X_train, y_train)

y_pred = pipe_knn.predict(X_test)
y_prob = pipe_knn.predict_proba(X_test)[:,1]

print("Baseline KNN")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Baseline KNN
Accuracy: 0.9831568474166726
Precision: 0.7962085308056872
Recall: 0.46408839779005523
F1: 0.5863874345549738
ROC-AUC: 0.8593325895115078


### KNN with Grid Search

In [4]:
param_grid_knn = {
    'knn__n_neighbors': [3, 5, 7, 9, 11],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan']
}

grid_knn = GridSearchCV(pipe_knn,param_grid_knn,cv=5,scoring='f1',n_jobs=-1)

grid_knn.fit(X_train, y_train)

best_knn = grid_knn.best_estimator_
y_pred = best_knn.predict(X_test)
y_prob = best_knn.predict_proba(X_test)[:,1]

print("\nKNN with Grid Search")
print("Best params:", grid_knn.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


KNN with Grid Search
Best params: {'knn__metric': 'euclidean', 'knn__n_neighbors': 7, 'knn__weights': 'distance'}
Accuracy: 0.9830147111079526
Precision: 0.8170103092783505
Recall: 0.43784530386740333
F1: 0.5701438848920863
ROC-AUC: 0.8708273368827752


### Knn with Dynamic Time Warping (DTW)

In [7]:
groups = {}

for col in TW_500.columns:
    if "_" in col and col != "label":
        parts = col.split("_")
        if len(parts) == 2 and parts[1].isdigit():
            prefix = parts[0]
            groups.setdefault(prefix, []).append(col)

for key in groups:
    groups[key] = sorted(groups[key], key=lambda x: int(x.split("_")[1]))

series_list = []

for prefix in groups:
    series = TW_500[groups[prefix]].values
    series_list.append(series)

X = np.stack(series_list, axis=2)
y = TW_500["label"].values

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,stratify=y,random_state=42)

X_small = X_train[:3000]
y_small = y_train[:3000]

knn_dtw = KNeighborsTimeSeriesClassifier(n_neighbors=5,metric="dtw")

knn_dtw.fit(X_small, y_small)

y_pred = knn_dtw.predict(X_test)
y_prob = knn_dtw.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.97725819060479
Precision: 0.6
Recall: 0.34806629834254144
F1: 0.4405594405594406
ROC-AUC: 0.841128206900415


In [10]:
import pandas as pd

knn_500 = [
    {"Model": "KNN baseline", "Accuracy": 0.98316, "Precision": 0.7962, "Recall": 0.4641, "F1": 0.5864, "ROC-AUC": 0.8593},
    {"Model": "KNN tuned", "Accuracy": 0.98301, "Precision": 0.8170, "Recall": 0.4378, "F1": 0.5701, "ROC-AUC": 0.8708},
    {"Model": "KNN + DTW", "Accuracy": 0.97726, "Precision": 0.6000, "Recall": 0.3481, "F1": 0.4406, "ROC-AUC": 0.8411}
]

df_knn_500 = pd.DataFrame(knn_500)
df_knn_500

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,KNN baseline,0.98316,0.7962,0.4641,0.5864,0.8593
1,KNN tuned,0.98301,0.8170,0.4378,0.5701,0.8708
2,KNN + DTW,0.97726,0.6000,0.3481,0.4406,0.8411


Zo všetkých variantov KNN sa ako najlepší ukázal základný model, ktorý dosiahol najvyššiu hodnotu F1-miery a zabezpečil stabilnú rovnováhu medzi presnosťou a spätnou väzbou. Najhoršie výsledky dosiahol model využívajúci metriku DTW, ktorý mal výrazne nižšiu hodnotu F1-miery, čo svedčí o jeho neefektívnosti pre danú úlohu.

### 1000

### Baseline KNN

In [5]:
X = TW_1000.drop(columns=['label'])
y = TW_1000['label']

X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,stratify=y,random_state=42)

pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

pipe_knn.fit(X_train, y_train)

y_pred = pipe_knn.predict(X_test)
y_prob = pipe_knn.predict_proba(X_test)[:,1]

print("Baseline KNN")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Baseline KNN
Accuracy: 0.9949186269632577
Precision: 0.7555555555555555
Recall: 0.5787234042553191
F1: 0.655421686746988
ROC-AUC: 0.8870541898661893


### KNN with Grid Search

In [6]:
param_grid_knn = {
    'knn__n_neighbors': [3, 5, 7, 9, 11],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan']
}

grid_knn = GridSearchCV(pipe_knn,param_grid_knn,cv=5,scoring='f1',n_jobs=-1)

grid_knn.fit(X_train, y_train)

best_knn = grid_knn.best_estimator_
y_pred = best_knn.predict(X_test)
y_prob = best_knn.predict_proba(X_test)[:,1]

print("\nKNN with Grid Search")
print("Best params:", grid_knn.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


KNN with Grid Search
Best params: {'knn__metric': 'manhattan', 'knn__n_neighbors': 3, 'knn__weights': 'distance'}
Accuracy: 0.9948120247317177
Precision: 0.7405405405405405
Recall: 0.5829787234042553
F1: 0.6523809523809524
ROC-AUC: 0.8704354661264733


### Knn with Dynamic Time Warping (DTW)

In [8]:
groups = {}

for col in TW_1000.columns:
    if "_" in col and col != "label":
        parts = col.split("_")
        if len(parts) == 2 and parts[1].isdigit():
            prefix = parts[0]
            groups.setdefault(prefix, []).append(col)

for key in groups:
    groups[key] = sorted(groups[key], key=lambda x: int(x.split("_")[1]))

series_list = []

for prefix in groups:
    series = TW_1000[groups[prefix]].values
    series_list.append(series)

X = np.stack(series_list, axis=2)
y = TW_1000["label"].values

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,stratify=y,random_state=42)

X_small = X_train[:3000]
y_small = y_train[:3000]

knn_dtw = KNeighborsTimeSeriesClassifier(n_neighbors=5,metric="dtw")

knn_dtw.fit(X_small, y_small)

y_pred = knn_dtw.predict(X_test)
y_prob = knn_dtw.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.9926799801009167
Precision: 0.6746987951807228
Recall: 0.23829787234042554
F1: 0.3522012578616352
ROC-AUC: 0.8745812268560699


In [13]:
knn_1000 = [
    {"Model": "KNN baseline", "Accuracy": 0.99492, "Precision": 0.7556, "Recall": 0.5787, "F1": 0.6554, "ROC-AUC": 0.8871},
    {"Model": "KNN grid searching", "Accuracy": 0.99481, "Precision": 0.7405, "Recall": 0.5830, "F1": 0.6524, "ROC-AUC": 0.8704},
    {"Model": "KNN with DTW", "Accuracy": 0.99268, "Precision": 0.6747, "Recall": 0.2383, "F1": 0.3522, "ROC-AUC": 0.8746}
]

df_knn_1000 = pd.DataFrame(knn_1000)
df_knn_1000

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,KNN baseline,0.99492,0.7556,0.5787,0.6554,0.8871
1,KNN grid searching,0.99481,0.7405,0.5830,0.6524,0.8704
2,KNN with DTW,0.99268,0.6747,0.2383,0.3522,0.8746


Zo všetkých variantov KNN sa opäť ako najlepší ukázal základný model. Najhoršie výsledky dosiahla model využívajúci metriku DTW

### 1500

### Baseline KNN

In [9]:
X = TW_1500.drop(columns=['label'])
y = TW_1500['label']

X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,stratify=y,random_state=42)

pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

pipe_knn.fit(X_train, y_train)

y_pred = pipe_knn.predict(X_test)
y_prob = pipe_knn.predict_proba(X_test)[:,1]

print("Baseline KNN")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Baseline KNN
Accuracy: 0.997761353137659
Precision: 0.7868852459016393
Recall: 0.4897959183673469
F1: 0.6037735849056604
ROC-AUC: 0.9019476318554808


###  KNN with Grid Search

In [10]:
param_grid_knn = {
    'knn__n_neighbors': [3, 5, 7, 9, 11],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan']
}

grid_knn = GridSearchCV(pipe_knn,param_grid_knn,cv=5,scoring='f1',n_jobs=-1)

grid_knn.fit(X_train, y_train)

best_knn = grid_knn.best_estimator_
y_pred = best_knn.predict(X_test)
y_prob = best_knn.predict_proba(X_test)[:,1]

print("\nKNN with Grid Search")
print("Best params:", grid_knn.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


KNN with Grid Search
Best params: {'knn__metric': 'euclidean', 'knn__n_neighbors': 3, 'knn__weights': 'distance'}
Accuracy: 0.997654750906119
Precision: 0.7222222222222222
Recall: 0.5306122448979592
F1: 0.611764705882353
ROC-AUC: 0.8512008461921354


### Knn with Dynamic Time Warping (DTW)

In [11]:
groups = {}

for col in TW_1500.columns:
    if "_" in col and col != "label":
        parts = col.split("_")
        if len(parts) == 2 and parts[1].isdigit():
            prefix = parts[0]
            groups.setdefault(prefix, []).append(col)

for key in groups:
    groups[key] = sorted(groups[key], key=lambda x: int(x.split("_")[1]))

series_list = []

for prefix in groups:
    series = TW_1500[groups[prefix]].values
    series_list.append(series)

X = np.stack(series_list, axis=2)
y = TW_1500["label"].values

X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2,stratify=y,random_state=42)

X_small = X_train[:3000]
y_small = y_train[:3000]

knn_dtw = KNeighborsTimeSeriesClassifier(n_neighbors=5,metric="dtw")

knn_dtw.fit(X_small, y_small)

y_pred = knn_dtw.predict(X_test)
y_prob = knn_dtw.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.9966242626678985
Precision: 1.0
Recall: 0.030612244897959183
F1: 0.0594059405940594
ROC-AUC: 0.6112273642876064


Zo všetkých variantov KNN sa opäť ako najlepší ukázal základný model. Najhoršie výsledky dosiahla model využívajúci metriku DTW.

In [12]:
knn_1500 = [
    {"Model": "KNN baseline", "Accuracy": 0.99776, "Precision": 0.7869, "Recall": 0.4898, "F1": 0.6038, "ROC-AUC": 0.9019},
    {"Model": "KNN grid searching", "Accuracy": 0.99765, "Precision": 0.7222, "Recall": 0.5306, "F1": 0.6118, "ROC-AUC": 0.8512},
    {"Model": "KNN with DTW", "Accuracy": 0.99662, "Precision": 1.0000, "Recall": 0.0306, "F1": 0.0594, "ROC-AUC": 0.6112}
]

df_knn_1500 = pd.DataFrame(knn_1500)
df_knn_1500 

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,KNN baseline,0.99776,0.7869,0.4898,0.6038,0.9019
1,KNN grid searching,0.99765,0.7222,0.5306,0.6118,0.8512
2,KNN with DTW,0.99662,1.0000,0.0306,0.0594,0.6112
